# Hausse de loyer 2026 - Collection Équinoxe

**Objectif :** estimer la hausse de loyer 2026 des six immeubles de Collection Équinoxe
(Daniel-Johnson, Lévesque, Saint-Elzéar, Le Carlyle, Westpark au Québec; The Met en Ontario),
à partir de l'historique réel des baux, avec une méthode expliquée et validée par backtest.

**Réponse en une ligne :** voir la section 1 pour la définition retenue et la section finale pour l'estimation.

## 0. Configuration

Toutes les constantes du notebook sont déclarées ici, une seule fois, avec leur justification.
Aucun nombre n'apparaît « en dur » plus bas : si un seuil change, il change ici.

Les fichiers CSV sources ne sont jamais modifiés; toute transformation se fait en mémoire dans ce notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# --- Fichiers sources (lecture seule) ---
LISTINGS_PATH = "equinoxe_listings.csv"
LEASES_PATH = "equinoxe_lease_history.csv"
CONCESSIONS_PATH = "equinoxe_concessions.csv"
ASKING_PATH = "equinoxe_asking_history.csv"

# --- Clé d'unité ---
# sPropCode + sUnitCode identifie un appartement de façon unique.
# Jamais sSite + sUnitCode : 248 des 1 061 unités entrent en collision.
UNIT_KEY = ["sPropCode", "sUnitCode"]

# --- Appariement de baux consécutifs d'une même unité ---
MIN_GAP_YEARS = 0.5    # écart plus court : généralement le même bail enregistré deux fois
MAX_GAP_YEARS = 2.5    # écart plus long : comparaison trop ancienne pour être équivalente
DAYS_PER_YEAR = 365.25

# --- Codes de sRenewal ---
RENEWAL = 1            # locataire en place qui renouvelle
TURNOVER = 0           # unité libérée puis relouée (relocation)

# --- Périodes ---
FIRST_ANALYSIS_YEAR = 2019   # avant 2019, trop peu de paires renouvellement pour être fiables
LAST_OBSERVED_YEAR = 2025    # les données s'arrêtent au 2025-12-31
FORECAST_YEAR = LAST_OBSERVED_YEAR + 1
BACKTEST_YEARS = [2023, 2024, 2025]

PCT = 100              # conversion ratio -> pourcentage

# --- Analyse des données (section 2) ---
RENT_MATCH_TOLERANCE = 1.0   # $ / mois : deux loyers « concordent » à un arrondi au dollar près
PROMO_CHARGE_CODE = "PromoPay"
TWO_BEDROOMS = 2
DOWNTIME_BINS_DAYS = [0, 7, 30, 60, np.inf]   # tranches de jours entre deux baux d'une même unité

### Chargement et conversion des dates

On charge les quatre tables, puis on convertit les colonnes de dates une seule fois.
`lease_start_year` (année de `sLeaseFrom`) est l'année à laquelle on rattache un bail :
c'est la date où le nouveau loyer **entre en vigueur**, donc celle qui compte pour les revenus.

In [ ]:
listings = pd.read_csv(LISTINGS_PATH)
leases = pd.read_csv(LEASES_PATH)
concessions = pd.read_csv(CONCESSIONS_PATH)
asking = pd.read_csv(ASKING_PATH)

for name, df in [("listings", listings), ("leases", leases),
                 ("concessions", concessions), ("asking", asking)]:
    print(f"{name:<12} {len(df):>6,} lignes   {df.shape[1]:>2} colonnes")

LEASE_DATE_COLUMNS = ["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"]
for col in LEASE_DATE_COLUMNS:
    leases[col] = pd.to_datetime(leases[col], errors="coerce")

leases["lease_start_year"] = leases["sLeaseFrom"].dt.year

for col in ["sDateFrom", "sDateTo"]:
    concessions[col] = pd.to_datetime(concessions[col])
asking["month_start"] = pd.to_datetime(asking["sMonth"] + "-01")

## 1. Définir la hausse mesurée

« La hausse de loyer 2026 » n'est pas un seul chiffre. Selon ce qu'on mesure, on obtient des réponses très différentes :

| Définition candidate | Ce qu'elle répond |
|---|---|
| Médiane annuelle de tous les baux | Combien coûte le bail « typique » cette année - **mesure surtout le mix d'immeubles** |
| Même unité, loyer contractuel (`sRent`) | De combien le prix inscrit au bail a monté pour un même appartement |
| Même unité, loyer effectif (`sRentEffective`) | De combien l'argent réellement perçu a monté pour un même appartement |
| Renouvellements seulement (`sRenewal = 1`) | Hausse consentie aux locataires en place - encadrée par le TAL / l'Ontario |
| Relocations seulement (`sRenewal = 0`) | Hausse du prix de marché lorsqu'une unité change de locataire |

Pour choisir, on calcule d'abord toutes ces mesures sur l'historique.

### Croissance à unité constante

La fonction ci-dessous compare chaque bail au **bail précédent de la même unité** (clé `UNIT_KEY`),
ne garde que les écarts comparables (entre `MIN_GAP_YEARS` et `MAX_GAP_YEARS`) et annualise la variation
pour que deux baux séparés de 18 mois soient comparables à deux baux séparés de 12 mois.

In [ ]:
def same_unit_growth(lease_df, price_col, date_col):
    """Croissance annualisée (%) du loyer, chaque unité comparée à son propre bail précédent.

    Retourne une ligne par paire (bail précédent -> bail courant) avec la colonne `growth_pct`.
    """
    ordered = lease_df.sort_values(UNIT_KEY + [date_col]).copy()
    by_unit = ordered.groupby(UNIT_KEY)
    ordered["prev_price"] = by_unit[price_col].shift(1)
    ordered["prev_date"] = by_unit[date_col].shift(1)
    pairs = ordered.dropna(subset=["prev_price", "prev_date"])

    pairs = pairs.assign(gap_years=(pairs[date_col] - pairs["prev_date"]).dt.days / DAYS_PER_YEAR)
    is_comparable_gap = pairs["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    has_positive_prices = (pairs["prev_price"] > 0) & (pairs[price_col] > 0)
    pairs = pairs[is_comparable_gap & has_positive_prices].copy()

    price_ratio = pairs[price_col] / pairs["prev_price"]
    pairs["growth_pct"] = (price_ratio ** (1 / pairs["gap_years"]) - 1) * PCT
    return pairs


contract_pairs = same_unit_growth(leases, price_col="sRent", date_col="sLeaseFrom")
effective_pairs = same_unit_growth(leases, price_col="sRentEffective", date_col="sLeaseFrom")
print(f"{len(contract_pairs):,} paires comparables sur {contract_pairs.groupby(UNIT_KEY).ngroups:,} unités")

### Tableau comparatif des définitions candidates

Médiane par année de début de bail. La médiane plutôt que la moyenne : quelques paires extrêmes
(relocation après une longue occupation, saisie atypique) ne doivent pas déplacer le chiffre.

In [ ]:
analysis_years = range(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR + 1)


def median_growth_by_year(pairs, renewal_flag=None):
    subset = pairs if renewal_flag is None else pairs[pairs["sRenewal"] == renewal_flag]
    return subset.groupby("lease_start_year")["growth_pct"].median()


naive_median_rent = leases.groupby("lease_start_year")["sRent"].median()

candidate_definitions = pd.DataFrame({
    "naive_median_all_leases": naive_median_rent.pct_change() * PCT,
    "same_unit_contract": median_growth_by_year(contract_pairs),
    "same_unit_effective": median_growth_by_year(effective_pairs),
    "renewal_contract": median_growth_by_year(contract_pairs, RENEWAL),
    "renewal_effective": median_growth_by_year(effective_pairs, RENEWAL),
    "turnover_contract": median_growth_by_year(contract_pairs, TURNOVER),
    "turnover_effective": median_growth_by_year(effective_pairs, TURNOVER),
}).loc[analysis_years].round(2)

candidate_definitions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for column, style in [("naive_median_all_leases", "o:"),
                      ("same_unit_contract", "o-"),
                      ("same_unit_effective", "s-")]:
    ax.plot(candidate_definitions.index, candidate_definitions[column], style, lw=2, label=column)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("année de début du bail")
ax.set_ylabel("croissance annuelle des loyers (%)")
ax.set_title("Trois définitions de « la hausse », trois histoires différentes")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Définition retenue

> **Hausse 2026 = croissance annualisée du loyer effectif (`sRentEffective`) à unité constante,
> médiane de tous les baux débutant en 2026 (renouvellements et relocations confondus),
> pour les six immeubles.**

**Pourquoi chacun de ces choix :**

1. **À unité constante, pas la médiane annuelle.** Le tableau montre que la médiane naïve s'emballe en 2023
   alors que la croissance à unité constante ralentit cette année-là : la médiane mesure l'arrivée des immeubles
   haut de gamme (Le Carlyle, Westpark, The Met), pas une hausse payée par les locataires. Comparer chaque
   appartement à lui-même neutralise la composition.

2. **Loyer effectif, pas contractuel.** La question d'affaires est de *budgéter les revenus*. Or les revenus sont
   ce qui est réellement perçu après les mois gratuits, le stationnement offert, etc. Les deux mesures évoluent
   ensemble jusqu'en 2023, puis divergent : en 2025, la croissance contractuelle est d'environ 7 % mais
   la croissance effective d'environ 3,6 %. Citer le contractuel à un propriétaire surestimerait la hausse
   de ses revenus de plus de trois points.

3. **Renouvellements et relocations confondus, mais prévus séparément.** Les revenus 2026 proviendront des deux
   types de baux, donc le chiffre final les combine. Mais ils obéissent à des règles différentes
   (plafond TAL / ligne directrice de l'Ontario pour les renouvellements, marché pour les relocations) et
   leur écart change de signe sur la période. On les modélisera donc séparément (section 6), puis on les
   recombinera selon la part attendue de chacun en 2026.

4. **Médiane annualisée, rattachée à l'année de `sLeaseFrom`.** La médiane est robuste aux paires atypiques;
   l'annualisation rend comparables des baux de durées différentes; `sLeaseFrom` est la date où le nouveau
   loyer s'applique.

**Indicateurs secondaires, rapportés à côté du chiffre principal :**

- **Renouvellements, loyer contractuel** : c'est le loyer contractuel que le TAL et la ligne directrice de l'Ontario
  encadrent. C'est donc la bonne mesure pour comparer aux taux réglementaires et pour fixer les loyers de renouvellement.
- **Relocations, loyer contractuel et effectif** : c'est la référence pour fixer les loyers affichés.
- **Par province** : The Met (Ontario) n'est jamais soumis au plafond québécois, ni l'inverse.

**Hypothèses à valider dans les sections suivantes :**

- `sRentEffective` étale correctement les concessions sur la durée du bail : **vérifié en section 2d**.
- Une paire de baux espacés de 6 à 30 mois compare bien le même produit (aucune rénovation majeure entre les deux).

## 2. Analyse des données

Avant de mesurer quoi que ce soit, on vérifie ce que les colonnes représentent **réellement** : les descriptions
ne correspondent pas toujours aux données. Chaque sous-section teste une hypothèse et se termine par une
**lecture** (ce que le résultat veut dire) et une **conséquence** (ce qu'on en fait).

Le travail complet est dans `exploration.ipynb`; on ne reprend ici que les vérifications qui changent une décision.
On garde les noms Yardi d'origine pendant les vérifications, et on renomme les colonnes à la fin de la section (2h).

### 2a. La clé d'unité

On vérifie quelles colonnes identifient un appartement de façon unique.

In [ ]:
hunits_per_key = leases.groupby(UNIT_KEY)["hUnit"].nunique().max()
keys_per_hunit = leases.groupby("hUnit")[UNIT_KEY].nunique().max().max()
keys_per_link = leases.groupby("sLink")[UNIT_KEY].nunique().max().max()
site_collisions = listings[listings.duplicated(["sSite", "sUnitCode"], keep=False)]
orphan_leases = (leases[UNIT_KEY].drop_duplicates()
                 .merge(listings[UNIT_KEY], how="left", indicator=True)
                 .query("_merge == 'left_only'"))

print(f"hUnit par clé sPropCode+sUnitCode (max)   : {hunits_per_key}")
print(f"clés par hUnit / par sLink (max)          : {keys_per_hunit} / {keys_per_link}")
print(f"collisions sur sSite + sUnitCode          : {len(site_collisions)} / {len(listings)} unités")
print(f"unités des baux absentes de listings      : {len(orphan_leases)}")
print(f"baux en double (même unité, même début)   : {leases.duplicated(UNIT_KEY + ['sLeaseFrom']).sum()}")

**Lecture :** `sPropCode + sUnitCode`, `hUnit` et `sLink` sont trois clés équivalentes (relation 1 pour 1).
`sSite + sUnitCode` fusionnerait 248 appartements distincts : un même site regroupe plusieurs propriétés
dont les numéros d'unité se chevauchent. Aucun bail orphelin, aucun doublon.

**Conséquence :** tout appariement se fait sur `UNIT_KEY = ["sPropCode", "sUnitCode"]`.

### 2b. Ce que représentent les colonnes des baux

Quatre colonnes dont le nom est trompeur ou le sens incertain :

- `sAvailable` : date de disponibilité, ou autre chose?
- `sTermSeq` : rang du bail du **locataire** (donc ≥ 2 = renouvellement) ou de l'**unité**?
- `sConcession` : information indépendante, ou simple indicateur dérivé des loyers?
- `sTermMonths` / `sLeaseTerm` : les baux font-ils tous 12 mois, et l'étiquette est-elle fiable?

In [ ]:
share_available_equals_start = (leases["sAvailable"] == leases["sLeaseFrom"]).mean()
print(f"sAvailable == sLeaseFrom : {share_available_equals_start:.1%} des baux\n")

print("sTermSeq x sRenewal :")
print(pd.crosstab(leases["sTermSeq"], leases["sRenewal"]), end="\n\n")

print("sConcession x (sRentEffective < sRent) :")
print(pd.crosstab(leases["sConcession"], leases["sRentEffective"] < leases["sRent"],
                  colnames=["sRentEffective < sRent"]), end="\n\n")

print("sLeaseTerm x sTermMonths :")
print(pd.crosstab(leases["sLeaseTerm"], leases["sTermMonths"]))

**Lecture :**

| Colonne | Constat | Ce qu'elle est réellement |
|---|---|---|
| `sAvailable` | identique à `sLeaseFrom` dans 100 % des baux | une copie de la date de début, sans information propre |
| `sTermSeq` | `sTermSeq = 1` : 1 061 baux, tous des relocations (= nombre d'unités); à partir de 2, les deux types | le **rang du bail dans l'historique de l'unité**, pas du locataire |
| `sConcession` | vaut 1 exactement quand `sRentEffective < sRent` | un indicateur **dérivé** des deux loyers |
| `sTermMonths` | 95 % à 12 mois, le reste à 5,9 / 17,9 / 23,9 mois | une durée calculée à partir des dates, fiable |
| `sLeaseTerm` | « 6 Months » contient des baux de 12 mois, « Short Term » uniquement des baux de 12 mois | une étiquette commerciale, **non fiable** |

**Conséquences :**
- renouvellements et relocations sont séparés avec `sRenewal`, jamais avec `sTermSeq`;
- `sConcession` n'est jamais utilisé comme variable à côté de `sRentEffective` (ce serait la même information deux fois);
- les variations de loyer sont **annualisées** avec les dates réelles, puisque les baux ne durent pas tous 12 mois;
- `sAvailable` (baux) et `sLeaseTerm` sont ignorés.

### 2c. Les concessions et le piège `PromoPay`

La table des concessions décrit `PromoPay` comme un montant **mensuel**. On compare, pour chaque code,
le montant typique (`sAmount`) et la durée (`sMonths`), puis on rapporte `PromoPay` au loyer du bail concerné.

Pour cela, on rattache chaque concession au bail de la même unité dont la période contient le début de la concession.

In [ ]:
def attach_concessions_to_leases(concession_df, lease_df):
    """Rattache chaque concession au bail de la même unité dont la période contient sDateFrom."""
    lease_columns = UNIT_KEY + ["sLeaseFrom", "sLeaseTo", "sRent"]
    candidates = (concession_df.reset_index(names="concession_id")
                  .merge(lease_df[lease_columns].reset_index(names="lease_id"), on=UNIT_KEY))
    starts_within_lease = candidates["sDateFrom"].between(candidates["sLeaseFrom"], candidates["sLeaseTo"])
    return candidates[starts_within_lease]


matched_concessions = attach_concessions_to_leases(concessions, leases)
print(f"concessions rattachées à un bail : {len(matched_concessions)} / {len(concessions)}\n")

concessions_by_code = (concessions.groupby("sChargeCode")
                       .agg(count=("sAmount", "size"),
                            median_amount=("sAmount", "median"),
                            median_months=("sMonths", "median"),
                            share_single_month=("sMonths", lambda months: (months == 1).mean()))
                       .sort_values("count", ascending=False))
print(concessions_by_code.round(2), end="\n\n")

promo = matched_concessions[matched_concessions["sChargeCode"] == PROMO_CHARGE_CODE]
promo_in_months_of_rent = -promo["sAmount"] / promo["sRent"]
print("PromoPay exprimé en mois de loyer :")
print(promo_in_months_of_rent.describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(promo_in_months_of_rent, bins=40)
ax.set_xlabel("montant PromoPay / loyer mensuel du bail (en mois de loyer)")
ax.set_ylabel("nombre de concessions")
ax.set_title("PromoPay : un montant unique proche d'un mois de loyer, pas un rabais mensuel")
plt.tight_layout()
plt.show()

**Lecture :** deux familles de concessions.

- **`PromoPay`** est inscrit sur un seul mois dans 88 % des cas, pour un montant médian de 0,8 mois de loyer, avec un pic
  à exactement 1 mois. Si c'était un rabais **mensuel**, les résidents habiteraient presque gratuitement : la description est
  fausse. C'est un **rabais unique de signature**, l'équivalent d'un « mois gratuit ».
- **Tous les autres codes** valent environ 130 à 150 $ par mois, sur 3, 6 ou 12 mois : ce sont de vrais rabais mensuels.
  `freepark`, `freelock` et `freeappl` portent sur des **accessoires** (stationnement, casier, électroménagers), pas sur le loyer.

Les 204 concessions non rattachées commencent toutes 1 à 30 jours avant un bail de la même unité (accordées entre la
signature et l'emménagement - détail dans `exploration.ipynb`).

**Conséquence :** `PromoPay` est renommé `one_time_signing_discount`. Son effet sur le loyer doit être **étalé sur la durée
du bail** (un mois gratuit sur 12 mois ≈ 8 % de rabais), ce qui est précisément ce que `sRentEffective` doit faire : on le vérifie en 2d.

### 2d. Validation de `sRentEffective`

Notre définition de la hausse (section 1) repose sur `sRentEffective`. On teste l'hypothèse :

> loyer effectif = loyer contractuel + (somme des concessions du bail, `sAmount × sMonths`) / durée du bail

Les montants de concessions sont négatifs, donc le loyer effectif est inférieur au loyer contractuel.

In [ ]:
concession_totals = (matched_concessions
                     .assign(total_amount=matched_concessions["sAmount"] * matched_concessions["sMonths"])
                     .groupby("lease_id")
                     .agg(concession_total=("total_amount", "sum"),
                          concession_count=("total_amount", "size")))

reconstruction = leases.join(concession_totals).fillna({"concession_total": 0, "concession_count": 0})
observed_gap = reconstruction["sRentEffective"] - reconstruction["sRent"]
predicted_gap = reconstruction["concession_total"] / reconstruction["sTermMonths"]
reconstruction["is_reconstructed"] = (observed_gap - predicted_gap).abs() < RENT_MATCH_TOLERANCE

print(f"sRentEffective reconstruit exactement : {reconstruction['is_reconstructed'].mean():.1%} des baux\n")

failures = reconstruction[~reconstruction["is_reconstructed"]]
print("Baux non reconstruits :")
print(pd.crosstab(observed_gap[failures.index] < 0, failures["concession_count"] > 0,
                  rownames=["rabais observé"], colnames=["concessions dans la table"]), end="\n\n")

reconstruction_rate_by_year = reconstruction.groupby("lease_start_year")["is_reconstructed"].mean() * PCT
print("% reconstruit par année :", reconstruction_rate_by_year.round(1).to_dict())

**Lecture :** la formule reconstruit `sRentEffective` **au dollar près pour environ 62 % des baux**. Une concordance exacte
sur des milliers de baux n'est pas une coïncidence : `sRentEffective` est bien le loyer contractuel diminué des concessions
étalées sur la durée du bail, `PromoPay` compris. Les échecs viennent de la **table des concessions**, pas de `sRentEffective` :

| Cas | Explication |
|---|---|
| Rabais observé, mais aucune concession dans la table (≈ 500 baux) | la table des concessions est un extrait incomplet |
| Concessions dans la table, mais aucun rabais (≈ 260 baux) | surtout des accessoires (`freepark`, `Freeothe`, `freelock`) non déduits du loyer |
| Les deux existent sans concorder (≈ 870 baux) | rattachement imparfait (voir 2c) |

Le taux de reconstruction baisse quand les concessions se multiplient (2024) : plus un bail en a, plus il y a de chances
qu'il en manque une dans l'extrait.

**Conséquence :** on utilise `sRentEffective` **tel que calculé par Yardi**, plutôt que de le recalculer à partir d'une table
incomplète. La définition de la section 1 est confirmée.

### 2e. L'effet de composition (mix)

Pourquoi la médiane annuelle de tous les baux ne mesure pas la hausse : on regarde *qui* est loué chaque année.

In [ ]:
analysis_window = leases[leases["lease_start_year"].between(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR)].copy()
analysis_window["rent_per_sqft"] = analysis_window["sRent"] / analysis_window["sSqft"]

mix_by_year = analysis_window.groupby("lease_start_year").agg(
    leases=("sRent", "size"),
    buildings=("sBuilding", "nunique"),
    median_rent=("sRent", "median"),
    median_sqft=("sSqft", "median"),
    pct_two_bed_plus=("sBeds", lambda beds: (beds >= TWO_BEDROOMS).mean() * PCT),
    median_rent_per_sqft=("rent_per_sqft", "median"))
mix_by_year.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

share_by_building = pd.crosstab(analysis_window["lease_start_year"], analysis_window["sBuilding"],
                                normalize="index") * PCT
share_by_building.plot(kind="bar", stacked=True, ax=axes[0], width=0.85)
axes[0].set_ylabel("% des baux de l'année")
axes[0].set_xlabel("année de début du bail")
axes[0].set_title("Trois immeubles haut de gamme entrent en 2023-2024")
axes[0].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))

rent_per_sqft_by_building = analysis_window.groupby("sBuilding")["rent_per_sqft"].median().sort_values()
rent_per_sqft_by_building.plot(kind="barh", ax=axes[1])
axes[1].set_xlabel("loyer médian au pi² ($)")
axes[1].set_title("Les nouveaux immeubles sont beaucoup plus chers au pi²")
plt.tight_layout()
plt.show()

**Lecture :** jusqu'en 2022, seuls les trois immeubles de Laval génèrent des baux. Le Carlyle et The Met entrent en 2023,
Westpark en 2024. Les unités louées deviennent **plus petites** (superficie médiane de 1 175 à 1 025 pi²) et comptent moins
de 2 chambres et plus (de 78 % à 60 %) - ce qui devrait faire *baisser* la médiane - et pourtant elle bondit de 10,8 % en 2023.
C'est que les nouveaux immeubles sont beaucoup plus chers au pi² : la médiane mélange des produits différents.
Pendant ce temps, un même appartement n'a augmenté que d'environ 2,3 % (section 1).

**Conséquence :** la médiane annuelle mesure la **composition du portefeuille**, pas la hausse des loyers. On compare chaque
unité à elle-même (sections 1 et 3).

**Limite à garder en tête :** avant 2023, l'historique ne couvre que Laval. Le backtest 2023 prédit donc un portefeuille
qui contient Mont-Royal et Ottawa avec un historique uniquement lavallois.

### 2f. Les concessions dans le temps

Rabais mesuré comme `1 - sRentEffective / sRent` : la part du loyer contractuel qui n'est pas perçue.

In [ ]:
analysis_window["discount_pct"] = (1 - analysis_window["sRentEffective"] / analysis_window["sRent"]) * PCT
analysis_window["lease_type"] = analysis_window["sRenewal"].map({RENEWAL: "renewal", TURNOVER: "turnover"})

concessions_by_year = analysis_window.pivot_table(index="lease_start_year", columns="lease_type",
                                                  values=["discount_pct", "sConcession"], aggfunc="mean")
concessions_by_year["sConcession"] *= PCT
concessions_by_year.round(2)

In [ ]:
discount_by_building_year = analysis_window.pivot_table(index="lease_start_year", columns="sBuilding",
                                                        values="discount_pct", aggfunc="mean")
fig, ax = plt.subplots(figsize=(10, 4.5))
discount_by_building_year.plot(ax=ax, marker="o")
ax.set_ylabel("rabais moyen (% du loyer contractuel)")
ax.set_xlabel("année de début du bail")
ax.set_title("Les concessions augmentent dans tous les immeubles, anciens comme nouveaux")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

**Lecture :** le rabais moyen passe d'environ 2 % (2019-2020) à environ 9 % (2025) et la part des baux avec concession
d'environ 50 % à plus de 80 %. Trois observations :

1. La hausse se produit **dans tous les immeubles**, y compris les anciens immeubles de Laval : ce n'est pas un effet
   de lancement des nouveaux immeubles, c'est un changement à l'échelle du portefeuille.
2. Les **renouvellements** reçoivent autant de concessions que les relocations (davantage en 2025) : JADCO doit
   concéder pour *retenir* ses locataires, pas seulement pour en attirer.
3. L'accélération commence en **2024** - l'année où croissance contractuelle et croissance effective divergent (section 1).

**Conséquence :** c'est l'explication de la divergence. En 2025, le loyer inscrit au bail monte d'environ 7 %, mais
JADCO en redonne une partie sous forme de concessions : le loyer perçu ne monte que d'environ 3,6 %.
Pour 2026, la question centrale devient l'**évolution des concessions** - que l'on relie au marché en section 6 (données externes).

### 2g. Test : les unités restées vides plus longtemps reçoivent-elles plus de concessions?

Intuition : un propriétaire qui peine à relouer offre des mois gratuits. Le test se fait **unité par unité**
(jours entre la fin d'un bail et le début du suivant). On n'en tire **aucun taux d'inoccupation** : l'extrait ne contient
qu'une partie des unités de chaque immeuble, un tel taux serait un artefact de l'extrait.

In [ ]:
leases_by_unit = leases.sort_values(UNIT_KEY + ["sLeaseFrom"]).copy()
leases_by_unit["days_between_leases"] = (leases_by_unit["sLeaseFrom"]
                                         - leases_by_unit.groupby(UNIT_KEY)["sLeaseTo"].shift(1)).dt.days
with_previous_lease = leases_by_unit.dropna(subset=["days_between_leases"]).copy()

print("Jours entre deux baux, selon le type du nouveau bail :")
print(with_previous_lease.groupby("sRenewal")["days_between_leases"].describe().round(1), end="\n\n")

turnovers = with_previous_lease[with_previous_lease["sRenewal"] == TURNOVER].copy()
turnovers["downtime_bucket"] = pd.cut(turnovers["days_between_leases"], bins=DOWNTIME_BINS_DAYS)
turnovers.groupby("downtime_bucket", observed=True).agg(
    leases=("sRent", "size"),
    pct_with_concession=("sConcession", lambda flag: flag.mean() * PCT)).round(1)

**Lecture :** résultat négatif.

- Le délai entre deux baux a **la même distribution** pour les renouvellements et les relocations (médiane de 1 jour).
  Or un renouvellement n'a, par définition, aucune période vide : ce délai ne mesure donc pas une vraie vacance.
- La part de relocations avec concession ne varie pas avec ce délai (environ 61-63 %); la seule tranche différente
  ne compte que 34 baux.

**Conséquence :** avec 2f (les renouvellements reçoivent autant de concessions), tout indique que les concessions sont une
**politique commerciale globale** liée au contexte de marché, pas une réaction à une unité restée vide. On teste donc la relation
avec le **taux d'inoccupation du marché publié par la SCHL** (section 6).

### 2h. Un signal interne pour 2026 : `sRent` de `listings`

`listings` contient une ligne par unité, avec sa propre colonne `sRent`. Ce n'est pas un loyer de bail : on la compare
au loyer du dernier bail de l'unité.

In [ ]:
last_lease_rent = (leases.sort_values("sLeaseFrom")
                   .groupby(UNIT_KEY).tail(1)
                   .set_index(UNIT_KEY)["sRent"])
listing_rent = listings.set_index(UNIT_KEY)["sRent"]
last_lease_rent = last_lease_rent.reindex(listing_rent.index)
equals_last_lease = (listing_rent - last_lease_rent).abs() < RENT_MATCH_TOLERANCE
listing_markup_pct = (listing_rent / last_lease_rent - 1) * PCT

print(f"sRent (listings) == loyer du dernier bail : {equals_last_lease.mean():.1%}")
print("Écart (%) entre sRent de listings et le loyer du dernier bail :")
print(listing_markup_pct.describe().round(2))

**Lecture :** `sRent` de `listings` n'est presque jamais égal au loyer du dernier bail, mais il est **toujours au-dessus**,
dans une fourchette étroite (0 % à environ +5 %, médiane environ +2,5 %). Hypothèse la plus probable : c'est le **loyer proposé
pour le prochain bail** de l'unité, en date de `sAsOf`.

**Conséquence :** c'est le seul regard vers l'avenir dans les données. On l'utilisera comme **point de comparaison** pour la
prévision 2026, pas comme prévision : c'est une hypothèse, un loyer proposé n'est pas un loyer signé, et il s'agit d'un loyer
contractuel, avant concessions.

### 2i. Renommage des colonnes

À partir d'ici, on renomme les colonnes selon ce qu'elles contiennent **réellement** (constats 2b à 2h).
Les clés (`sPropCode`, `sUnitCode`, `hUnit`) et les colonnes descriptives (`sBuilding`, `sState`, `sBeds`...) gardent leur nom
Yardi, qui est déjà exact. **Cette étape modifie les tables** : toutes les sections suivantes utilisent les nouveaux noms.
Les fichiers CSV sources ne sont pas modifiés.

In [ ]:
LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",            # contractuel - concessions étalées sur le bail (2d)
    "sConcession": "has_concession",               # = effective_rent < contract_rent (2b)
    "sRenewal": "is_renewal",                      # 1 = locataire en place, 0 = relocation
    "sTermSeq": "unit_lease_rank",                 # rang du bail dans l'historique de l'unité (2b)
    "sTermMonths": "lease_term_months",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sSignDate": "sign_date",
    "sAvailable": "lease_start_duplicate",         # copie exacte de sLeaseFrom (2b)
    "sLeaseTerm": "lease_term_label_unreliable",   # étiquette incohérente avec la durée réelle (2b)
}
LISTING_COLUMN_RENAMES = {
    "sRent": "proposed_next_rent",                 # hypothèse : loyer proposé au prochain bail (2h)
    "sAsOf": "proposed_rent_as_of",
    "sAvailable": "next_available_date",
}
ASKING_COLUMN_RENAMES = {"sAskingRent": "asking_rent_at_remarketing"}
CONCESSION_COLUMN_RENAMES = {
    "sAmount": "amount_per_month",
    "sMonths": "months_applied",
    "sDateFrom": "concession_start",
    "sDateTo": "concession_end",
}
CHARGE_CODE_LABELS = {
    "PromoPay": "one_time_signing_discount",       # PAS mensuel : environ un mois de loyer, une fois (2c)
    "freerent": "free_rent_monthly",
    "freepark": "free_parking_accessory",
    "freelock": "free_locker_accessory",
    "freeappl": "free_appliances_accessory",
    "Freeothe": "other_uncategorized",
    "Indem": "tenant_indemnity",
    "Referenc": "referral_credit",
}

leases = leases.rename(columns=LEASE_COLUMN_RENAMES)
listings = listings.rename(columns=LISTING_COLUMN_RENAMES)
asking = asking.rename(columns=ASKING_COLUMN_RENAMES)
concessions = concessions.rename(columns=CONCESSION_COLUMN_RENAMES)
concessions["charge_type"] = concessions["sChargeCode"].map(CHARGE_CODE_LABELS)

leases[list(LEASE_COLUMN_RENAMES.values())].head()

### Synthèse de la section 2

| Constat | Décision |
|---|---|
| `sPropCode + sUnitCode` = `hUnit` = `sLink`; `sSite + sUnitCode` fusionne 248 unités | appariement sur `UNIT_KEY` |
| `sTermSeq` = rang du bail de l'unité | types de baux séparés avec `is_renewal` |
| baux de 6 à 24 mois; `sLeaseTerm` incohérent | variations annualisées avec les dates réelles |
| `sConcession` dérivé des deux loyers | jamais utilisé comme variable à côté de `effective_rent` |
| `PromoPay` = mois gratuit unique, pas mensuel | renommé, étalé sur le bail |
| `sRentEffective` validé (62 % reconstruits exactement, écarts dus à la table des concessions) | **définition de la section 1 confirmée** |
| médiane annuelle tirée par 3 immeubles haut de gamme | croissance à unité constante |
| rabais moyen de 2 % à 9 %, partout, renouvellements compris | explique la divergence contractuel / effectif depuis 2024 |
| aucun lien interne vacance → concessions | inoccupation SCHL comme contexte (section 6) |
| `sRent` de `listings` ≈ +2,5 % au-dessus du dernier bail | point de comparaison pour 2026 |